# CS-781 Capstone, Phase 3: Rhetorical Role prediction with the Phase-1 Decoder-only Transformer (GQA + RoPE + SwiGLU, 121M)

One notebook, four presets (three main experiments + one optional extra), each producing its own `submission_<EXP>.zip`. Set `EXPERIMENT` in section 1 and run all cells; repeat for the next preset.

**Task:** label every sentence of a legal document with one of 13 rhetorical roles (CL and IT domains). Metric: macro-F1.

**How the model sees a document.** Both Phase-1 models are causal (left context only), so:
1. Sentences are tokenised with the Phase-1 tokenizer (after the same `normalize_text`) and packed, with a separator token after each, into **windows of whole sentences** (about 1024 tokens, 50% overlap).
2. The backbone runs over the window. A **sentence vector** = mean of the sentence's token states, concatenated with the state at its separator token (which has read the whole sentence).
3. A **BiGRU over the sentence vectors** gives each sentence both left and right document context (set `CONTEXT_MODE = "sentence"` to switch context off for an ablation).
4. Per-sentence log-probabilities from overlapping windows are averaged (sentences near a window edge get less weight), then the arg-max label is taken. Exactly one label per sentence is guaranteed by construction.

**Knowledge distillation is done at the sentence level.** The teacher only supplies a probability distribution over the 13 labels for each sentence, so it does not matter which tokenizer or architecture the teacher uses: any run of either notebook can teach any student. Every non-KD run saves a `teacher_<EXP>.pkl` for this purpose.

| Preset | Technique | Trainable |
|---|---|---|
| `D1_lorma` | **LORMA** (multiplicative low-rank, mandatory technique) | LORMA matrices + head |
| `D2_ia3` | IA3 (learned scaling vectors) | scaling vectors + head |
| `D3_kd` | Knowledge distillation into a half-depth student | whole student |
| `D4_lora` | (extra) LoRA | LoRA matrices + head |

**About LORMA (please read).** The assignment cites LORMA only as reference [12] and does not define it. The implementation here is my reconstruction from memory of the idea of *multiplicative* low-rank adaptation: the frozen weight `W0` is multiplied by a learned low-rank-perturbed identity, `W' = (I + s·B·A)·W0` (applied to the layer output) or `W' = W0·(I + s·B·A)` (applied to the layer input), with `B` zero-initialised so training starts from the Phase-1 function. `side="auto"` picks the cheaper side. I could not check the paper, so **compare this with reference [12] and edit `LoRMALinear` if the paper's formulation differs** (for example in how it avoids rank collapse). Do not describe this as a faithful reproduction in the report until that check is done.

## 1. Configuration

In [64]:
import os, sys, json, re, math, time, random, zipfile, heapq, shutil, subprocess, hashlib, pickle, glob
from collections import Counter, defaultdict
from dataclasses import dataclass
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.checkpoint

DEBUG = False                 # True = tiny synthetic smoke test
SEED = 42
EXPERIMENT = "D4_lora"

HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
# ---- Phase-1 assets: decoder-only checkpoint (best.pt from the 121M GQA model) and ITS OWN bpe_merges.json (the 88M-token-corpus tokenizer) ----
CKPT_PATH = "/kaggle/input/models/ashishkj93/phase-1-decoder-bundle/pytorch/default/1/best.pt"
TOK_PATH  = "/kaggle/input/models/ashishkj93/phase-1-decoder-bundle/pytorch/default/1/bpe_merges.json"
OUTPUT_DIR = "/kaggle/working/phase3_rr"
EOS_PLACEHOLDER = "<|endoftext|>"
# teacher file for the *_kd presets (written by any earlier non-KD run: <OUTPUT_DIR>/<EXP>/teacher_<EXP>.pkl)
TEACHER_PKL = "/kaggle/working/phase3_rr/D2_ia3/teacher_D2_ia3.pkl"

PRESETS = {
    "D1_lorma": dict(method="lorma", lr_backbone=1e-3, lr_head=1e-3, r=16, alpha=32, dropout=0.05, side="auto",
                     desc="LORMA: multiplicative low-rank update on every linear layer (backbone frozen)"),
    "D2_ia3":   dict(method="ia3",   lr_backbone=3e-3, lr_head=1e-3,
                     desc="IA3: learned scaling vectors on attention keys/values and FFN activations (backbone frozen)"),
    "D3_kd":    dict(method="full",  lr_backbone=1e-4, lr_head=1e-3, student_keep="odd", kd_alpha=0.7, kd_temp=2.0,
                     desc="Knowledge distillation: half-depth student (odd layers of the Phase-1 model) learns from teacher soft labels"),
    "D4_lora":  dict(method="lora",  lr_backbone=1e-3, lr_head=1e-3, r=16, alpha=32, dropout=0.05,
                     desc="(optional extra) LoRA r=16 on every linear layer"),
}

# ---- shared by all presets (the controlled part) ----
WINDOW_TOK, SENT_MAX_TOKENS, WIN_OVERLAP = 1024, 96, 0.5     # tokens per window (<= model context), cap per sentence, window overlap
CONTEXT_MODE = "document"     # "document" (window of sentences + BiGRU) | "sentence" (each sentence alone)
MICRO_BS, MICRO_BS_SENT, GRAD_ACCUM = 4, 32, 4
EPOCHS = 12
WARMUP_FRAC, WEIGHT_DECAY, CLIP = 0.06, 0.01, 1.0
SENT_PROJ, GRU_HIDDEN, HEAD_DROPOUT = 512, 256, 0.2
CLASS_WEIGHT = "sqrt_inv"     # "sqrt_inv" (rare roles up-weighted, frequent ones down-weighted, clipped) | "none"
GRAD_CKPT = False             # True = less memory, ~30% slower
FINAL_RETRAIN = True          # retrain on train+dev for the best epoch count, then predict the test set
MAX_TRAIN_MINUTES = 60        # per training phase; stops early and keeps the best epoch so far
SAVE_LOGPROBS = True          # save teacher_<EXP>.pkl (per-sentence log-probs) for later distillation / ensembling
ALLOW_LARGER_STUDENT = False  # KD sanity check: the student must have fewer parameters than the teacher

if DEBUG:
    WINDOW_TOK, SENT_MAX_TOKENS = 128, 24
    MICRO_BS, MICRO_BS_SENT, GRAD_ACCUM, EPOCHS, SENT_PROJ, GRU_HIDDEN = 4, 8, 1, 3, 32, 16
    MAX_TRAIN_MINUTES = 10

EXP = PRESETS[EXPERIMENT]
EXP_DIR = f"{OUTPUT_DIR}/{EXPERIMENT}"
os.makedirs(EXP_DIR, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device, "|", torch.cuda.get_device_name(0) if device == "cuda" else "", "| experiment:", EXPERIMENT, "-", EXP["desc"])

device: cuda | Tesla T4 | experiment: D4_lora - (optional extra) LoRA r=16 on every linear layer


## 2. Phase-1 code, reused unchanged
`normalize_text`, the from-scratch `ByteBPE` tokenizer and the model architecture, copied from the Phase-1 notebooks so the checkpoint loads exactly.

In [65]:
CHAR_MAP = str.maketrans({
    "\u2018": "'", "\u2019": "'", "\u201c": '"', "\u201d": '"', "\u2013": "-", "\u2014": "-",
    "\u2212": "-", "\u2026": "...", "\u00a0": " ", "\u200b": "", "\ufeff": "", "\u00ad": "",
})
CTRL = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]")

def normalize_text(t: str, keep_newlines: bool) -> str:
    """Character and whitespace normalisation. Applied to train text AND test prompts."""
    t = CTRL.sub(" ", t.translate(CHAR_MAP))
    if keep_newlines:
        t = re.sub(r"[ \t]+", " ", t)
        t = re.sub(r" ?\n ?", "\n", t)
        t = re.sub(r"\n{3,}", "\n\n", t)
    else:
        t = re.sub(r"\s+", " ", t)
    return t.strip()

PRETOKEN_RE = re.compile(r"""'(?:[sdmt]|ll|ve|re)| ?[^\W\d_]+| ?\d{1,3}| ?(?:[^\s\w]|_)+|\s+(?!\S)|\s+""")

class ByteBPE:
    def __init__(self, merges=None):
        self.merges = [tuple(m) for m in (merges or [])]
        self.rank = {p: i for i, p in enumerate(self.merges)}
        self.vocab = [bytes([i]) for i in range(256)]
        for a, b in self.merges:
            self.vocab.append(self.vocab[a] + self.vocab[b])
        self.eos_id = len(self.vocab)
        self.vocab.append(EOS_PLACEHOLDER.encode())
        self.vocab_size = len(self.vocab)
        self._cache = {}

    # ---------- training ----------
    @staticmethod
    def train(chunk_counts, vocab_size, min_freq=3, log_every=2000):
        n_merges = vocab_size - 256 - 1                      # last id is EOS
        items = [(c.encode("utf-8"), f) for c, f in chunk_counts.items() if f >= min_freq]
        words = [list(b) for b, _ in items]
        freqs = [f for _, f in items]
        pair_cnt, where = defaultdict(int), defaultdict(set)
        for i, w in enumerate(words):
            for p in zip(w, w[1:]):
                pair_cnt[p] += freqs[i]; where[p].add(i)
        heap = [(-c, p) for p, c in pair_cnt.items()]
        heapq.heapify(heap)
        merges, t0 = [], time.time()
        while len(merges) < n_merges:
            best = None
            while heap:                                       # lazy deletion
                negc, p = heapq.heappop(heap)
                c = pair_cnt.get(p, 0)
                if c <= 0: continue
                if c != -negc:
                    heapq.heappush(heap, (-c, p)); continue
                best = p; break
            if best is None: break
            a, b = best; new_id = 256 + len(merges); merges.append(best)
            changed = set()
            for i in list(where[best]):
                w, f = words[i], freqs[i]
                out, j, n, hit = [], 0, len(w), False
                while j < n:
                    if j < n - 1 and w[j] == a and w[j + 1] == b:
                        out.append(new_id); j += 2; hit = True
                    else:
                        out.append(w[j]); j += 1
                if not hit: continue
                for q in zip(w, w[1:]): pair_cnt[q] -= f
                for q in zip(out, out[1:]):
                    pair_cnt[q] += f; where[q].add(i); changed.add(q)
                words[i] = out
            where.pop(best, None); pair_cnt.pop(best, None)
            for q in changed:
                if pair_cnt[q] > 0: heapq.heappush(heap, (-pair_cnt[q], q))
            if len(merges) % log_every == 0:
                print(f"  merges {len(merges):6d}/{n_merges} | {time.time() - t0:5.0f}s")
        return ByteBPE(merges)

    # ---------- encoding / decoding ----------
    def _encode_chunk(self, chunk):
        hit = self._cache.get(chunk)
        if hit is not None: return hit
        ids = list(chunk.encode("utf-8"))
        while len(ids) > 1:
            best_r, best_p = None, None
            for p in zip(ids, ids[1:]):
                r = self.rank.get(p)
                if r is not None and (best_r is None or r < best_r):
                    best_r, best_p = r, p
            if best_p is None: break
            a, b = best_p; new_id = 256 + best_r; out, j = [], 0
            while j < len(ids):
                if j < len(ids) - 1 and ids[j] == a and ids[j + 1] == b:
                    out.append(new_id); j += 2
                else:
                    out.append(ids[j]); j += 1
            ids = out
        if len(self._cache) > 3_000_000: self._cache.clear()
        self._cache[chunk] = ids
        return ids

    def encode(self, text):
        ids = []
        for chunk in PRETOKEN_RE.findall(text):
            ids.extend(self._encode_chunk(chunk))
        return ids

    def decode(self, ids):
        data = b"".join(self.vocab[i] for i in ids if i != self.eos_id)
        return data.decode("utf-8", errors="replace")

    def save(self, path):
        json.dump({"merges": self.merges, "eos": EOS_PLACEHOLDER}, open(path, "w"))

    @staticmethod
    def load(path):
        return ByteBPE(json.load(open(path))["merges"])

In [66]:
@dataclass
class ModelConfig:
    vocab_size: int
    n_ctx: int = 1280
    n_layer: int = 12
    n_head: int = 12
    n_kv_head: int = 4
    n_embd: int = 768
    n_ffn: int = 2816
    rope_base: float = 10000.0

class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim)); self.eps = eps
    def forward(self, x):
        xf = x.float()
        return (xf * torch.rsqrt(xf.pow(2).mean(-1, keepdim=True) + self.eps)).type_as(x) * self.weight

def build_rope(head_dim, n_ctx, base):
    inv = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
    freqs = torch.outer(torch.arange(n_ctx).float(), inv)
    return freqs.cos(), freqs.sin()                          # (n_ctx, head_dim/2)

def apply_rope(x, cos, sin):                                 # x: (B, H, T, hd); cos/sin: (*, 1, T, hd/2)
    x1, x2 = x.float().chunk(2, dim=-1)
    return torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).type_as(x)

class Attention(nn.Module):
    def __init__(self, c: ModelConfig):
        super().__init__()
        self.nh, self.nkv, self.hd = c.n_head, c.n_kv_head, c.n_embd // c.n_head
        assert c.n_head % c.n_kv_head == 0
        self.q = nn.Linear(c.n_embd, self.nh * self.hd, bias=False)
        self.k = nn.Linear(c.n_embd, self.nkv * self.hd, bias=False)
        self.v = nn.Linear(c.n_embd, self.nkv * self.hd, bias=False)
        self.o = nn.Linear(self.nh * self.hd, c.n_embd, bias=False)
    def forward(self, x, cos, sin, mask=None, cache=None):
        B, T, _ = x.shape
        q = self.q(x).view(B, T, self.nh, self.hd).transpose(1, 2)
        k = self.k(x).view(B, T, self.nkv, self.hd).transpose(1, 2)
        v = self.v(x).view(B, T, self.nkv, self.hd).transpose(1, 2)
        q, k = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
        new_cache = None
        if cache is not None:                                # generation: grow the KV-cache
            if cache[0] is not None:
                k, v = torch.cat([cache[0], k], 2), torch.cat([cache[1], v], 2)
            new_cache = (k, v)
        rep = self.nh // self.nkv
        if rep > 1:
            k, v = k.repeat_interleave(rep, 1), v.repeat_interleave(rep, 1)
        y = F.scaled_dot_product_attention(q, k, v, attn_mask=mask,
                                           is_causal=(mask is None and cache is None))
        return self.o(y.transpose(1, 2).reshape(B, T, -1)), new_cache

class SwiGLU(nn.Module):
    def __init__(self, c: ModelConfig):
        super().__init__()
        self.gate = nn.Linear(c.n_embd, c.n_ffn, bias=False)
        self.up = nn.Linear(c.n_embd, c.n_ffn, bias=False)
        self.down = nn.Linear(c.n_ffn, c.n_embd, bias=False)
    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))

class Block(nn.Module):
    def __init__(self, c: ModelConfig):
        super().__init__()
        self.n1, self.attn = RMSNorm(c.n_embd), Attention(c)
        self.n2, self.mlp = RMSNorm(c.n_embd), SwiGLU(c)
    def forward(self, x, cos, sin, mask=None, cache=None):
        h, cache = self.attn(self.n1(x), cos, sin, mask, cache)
        x = x + h
        return x + self.mlp(self.n2(x)), cache

class LegalLM(nn.Module):
    def __init__(self, c: ModelConfig):
        super().__init__()
        self.c = c
        self.tok_emb = nn.Embedding(c.vocab_size, c.n_embd)
        self.blocks = nn.ModuleList([Block(c) for _ in range(c.n_layer)])
        self.norm = RMSNorm(c.n_embd)
        self.head = nn.Linear(c.n_embd, c.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight               # weight tying
        cos, sin = build_rope(c.n_embd // c.n_head, c.n_ctx, c.rope_base)
        self.register_buffer("cos", cos, persistent=False)
        self.register_buffer("sin", sin, persistent=False)
        self.grad_ckpt = False
        self.apply(self._init)
        for n, p in self.named_parameters():                 # GPT-2 style scaled residual init
            if n.endswith("attn.o.weight") or n.endswith("mlp.down.weight"):
                nn.init.normal_(p, 0.0, 0.02 / math.sqrt(2 * c.n_layer))
    @staticmethod
    def _init(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, 0.0, 0.02)

    def forward(self, idx, targets=None, pos_ids=None, mask=None, caches=None, last_only=False):
        B, T = idx.shape
        x = self.tok_emb(idx)
        if pos_ids is None:
            cos, sin = self.cos[:T][None, None], self.sin[:T][None, None]
        else:                                                # left-padded batches carry their own positions
            cos, sin = self.cos[pos_ids][:, None], self.sin[pos_ids][:, None]
        new_caches = [] if caches is not None else None
        for i, blk in enumerate(self.blocks):
            if self.grad_ckpt and self.training:
                x = torch.utils.checkpoint.checkpoint(
                    lambda x_, c_, s_, b=blk: b(x_, c_, s_)[0], x, cos, sin, use_reentrant=False)
            else:
                x, c_new = blk(x, cos, sin, mask, caches[i] if caches is not None else None)
                if new_caches is not None: new_caches.append(c_new)
        x = self.norm(x)
        if last_only: x = x[:, -1:]
        logits = self.head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.float().view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss, new_caches

## 3. Load the Phase-1 checkpoint and tokenizer

In [67]:
for p in (CKPT_PATH, TOK_PATH):
    if not os.path.exists(p):
        print("MISSING:", p, "\nFiles under /kaggle/input:")
        for root, _, files in os.walk("/kaggle/input"):
            for f in files: print("  ", os.path.join(root, f))
        raise FileNotFoundError(p)
ck = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
p1_cfg = ModelConfig(**ck["cfg"])
tokenizer = ByteBPE.load(TOK_PATH)
EOS_ID = tokenizer.eos_id
assert p1_cfg.vocab_size == tokenizer.vocab_size, \
    f"tokenizer/checkpoint mismatch: model vocab {p1_cfg.vocab_size} vs tokenizer {tokenizer.vocab_size}. Use the bpe_merges.json this checkpoint was trained with."
print(f"checkpoint step {ck.get('step')} | vocab {p1_cfg.vocab_size} | n_ctx {p1_cfg.n_ctx} | config: {ck['cfg']}")
print("tokenizer merges md5:", hashlib.md5(open(TOK_PATH, "rb").read()).hexdigest()[:12])

def build_backbone():
    m = LegalLM(p1_cfg)
    m.load_state_dict(ck["model"])
    return m

def backbone_dim(m):
    return m.c.n_embd

def run_blocks(m, x):
    T = x.size(1)
    cos, sin = m.cos[:T][None, None], m.sin[:T][None, None]
    for blk in m.blocks:
        if GRAD_CKPT and m.training:
            x = torch.utils.checkpoint.checkpoint(lambda x_, b=blk: b(x_, cos, sin, None, None)[0], x, use_reentrant=False)
        else:
            x, _ = blk(x, cos, sin, None, None)
    return m.norm(x)

def prune_layers(m, keep):
    """Keep only the listed layers (a half-depth student that still starts from Phase-1 weights)."""
    m.blocks = nn.ModuleList([m.blocks[i] for i in keep])
    return m

_probe = build_backbone().eval()
print(f"Phase-1 backbone: {sum(p.numel() for p in _probe.parameters()) / 1e6:.1f}M parameters, {len(_probe.blocks)} layers (loaded OK)")
del _probe

checkpoint step 1076 | vocab 32000 | n_ctx 1280 | config: {'vocab_size': 32000, 'n_ctx': 1280, 'n_layer': 12, 'n_head': 12, 'n_kv_head': 4, 'n_embd': 768, 'n_ffn': 2816, 'rope_base': 10000.0}
tokenizer merges md5: 8f09a8a79d88
Phase-1 backbone: 121.3M parameters, 12 layers (loaded OK)


## 4. Phase-3 data: load, parse robustly, audit
The loader accepts the common JSON layouts (dict of documents or list of records; labels as strings or integer ids; the label vocabulary as a list or a dict). The audit prints label counts per domain so class imbalance is visible before training.

In [68]:
from huggingface_hub import HfApi, hf_hub_download, login
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("capstone-phase1-token")
    os.environ["HF_TOKEN"] = HF_TOKEN
    login(token=HF_TOKEN)
except Exception:
    HF_TOKEN = os.environ.get("HF_TOKEN")
_repo_files = HfApi().list_repo_files(HF_DATASET_REPO, repo_type="dataset", token=HF_TOKEN)

def fetch(*must_contain):
    hits = [f for f in _repo_files if all(s.lower() in f.lower() for s in must_contain)]
    if not hits:
        raise FileNotFoundError(f"{must_contain} not found. Repo has: {_repo_files[:60]}")
    return hf_hub_download(HF_DATASET_REPO, hits[0], repo_type="dataset", token=HF_TOKEN)
print([f for f in _repo_files if "phase_3" in f.lower()])

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


['Phase_3_RR/Test/CL_test.json', 'Phase_3_RR/Test/IT_test.json', 'Phase_3_RR/Train/CL_dev.json', 'Phase_3_RR/Train/CL_train.json', 'Phase_3_RR/Train/IT_dev.json', 'Phase_3_RR/Train/IT_train.json', 'Phase_3_RR/Train/label_vocab.json']


In [69]:
P3 = {k: fetch("phase_3", k) for k in ["CL_train", "CL_dev", "IT_train", "IT_dev", "CL_test", "IT_test", "label_vocab"]}

def load_json(p):
    with open(p, encoding="utf-8") as f: return json.load(f)

vocab_obj = load_json(P3["label_vocab"])
if isinstance(vocab_obj, dict):
    if all(isinstance(v, int) for v in vocab_obj.values()): LABELS = [k for k, _ in sorted(vocab_obj.items(), key=lambda kv: kv[1])]
    elif all(str(k).isdigit() for k in vocab_obj): LABELS = [v for _, v in sorted(vocab_obj.items(), key=lambda kv: int(kv[0]))]
    else: LABELS = list(vocab_obj)
else:
    LABELS = [str(l) for l in vocab_obj]
LABELS = ["None" if l is None else str(l) for l in LABELS]
L2I = {l: i for i, l in enumerate(LABELS)}
K = len(LABELS)
print(f"{K} labels (exact strings from label_vocab.json): {LABELS}")

def lab_to_str(l):
    if l is None: return "None"
    if isinstance(l, (int, np.integer)): return LABELS[int(l)]
    return str(l)

def parse_docs(obj, domain, labelled):
    if isinstance(obj, dict) and ("text" in obj or "sentences" in obj): obj = [obj]
    if isinstance(obj, dict) and len(obj) == 1:
        only = next(iter(obj.values()))
        if isinstance(only, list) and only and isinstance(only[0], dict): obj = only
    items = [(str(r.get("id", r.get("doc_id", i))), r) for i, r in enumerate(obj)] if isinstance(obj, list) \
        else [(str(k), v) for k, v in obj.items()]
    docs = []
    for did, r in items:
        sents = r if isinstance(r, list) else r.get("text", r.get("sentences"))
        sents = [sents] if isinstance(sents, str) else [str(s) for s in sents]
        labs = None
        if labelled:
            raw = r.get("labels", r.get("label", r.get("roles"))) if isinstance(r, dict) else None
            assert raw is not None and len(raw) == len(sents), f"{domain} doc {did}: labels missing or length {None if raw is None else len(raw)} != sentences {len(sents)}"
            labs = [lab_to_str(l) for l in raw]
            bad = set(labs) - set(LABELS)
            assert not bad, f"{domain} doc {did}: labels not in label_vocab.json: {bad}"
        docs.append(dict(key=f"{domain}::{did}", id=did, domain=domain, raw_sents=sents, labels=labs))
    return docs

RAW = {"train": parse_docs(load_json(P3["CL_train"]), "CL", True) + parse_docs(load_json(P3["IT_train"]), "IT", True),
       "dev":   parse_docs(load_json(P3["CL_dev"]), "CL", True) + parse_docs(load_json(P3["IT_dev"]), "IT", True),
       "test":  parse_docs(load_json(P3["CL_test"]), "CL", False) + parse_docs(load_json(P3["IT_test"]), "IT", False)}

ids_cl = {d["id"] for d in RAW["test"] if d["domain"] == "CL"}; ids_it = {d["id"] for d in RAW["test"] if d["domain"] == "IT"}
if ids_cl & ids_it:
    raise ValueError(f"{len(ids_cl & ids_it)} test document IDs occur in BOTH CL_test and IT_test (e.g. {sorted(ids_cl & ids_it)[:3]}). "
                     "The submission maps one ID to one list, so check with the organisers how colliding IDs must be keyed before submitting.")

print("\nsplit  | domain | docs | sentences | median sents/doc | max sents/doc")
for sp in RAW:
    for dom in ("CL", "IT"):
        ds = [d for d in RAW[sp] if d["domain"] == dom]
        ns = [len(d["raw_sents"]) for d in ds] or [0]
        print(f"{sp:6s} | {dom}     | {len(ds):4d} | {sum(ns):9d} | {int(np.median(ns)):16d} | {max(ns)}")
cnt = {sp: Counter(l for d in RAW[sp] if d["labels"] for l in d["labels"]) for sp in ("train", "dev")}
print("\nlabel counts:  label | train | dev")
for l in LABELS: print(f"  {l:24s} {cnt['train'].get(l, 0):7d} {cnt['dev'].get(l, 0):7d}")
print("sample:", RAW["train"][0]["raw_sents"][:2], RAW["train"][0]["labels"][:2])

13 labels (exact strings from label_vocab.json): ['Fact', 'Issue', 'ArgumentPetitioner', 'ArgumentRespondent', 'Statute', 'Dissent', 'PrecedentReliedUpon', 'PrecedentNotReliedUpon', 'PrecedentOverruled', 'RulingByLowerCourt', 'RatioOfTheDecision', 'RulingByPresentCourt', 'None']

split  | domain | docs | sentences | median sents/doc | max sents/doc
train  | CL     |   40 |     10593 |              120 | 1816
train  | IT     |   40 |      6252 |              126 | 442
dev    | CL     |    5 |      1396 |              184 | 848
dev    | IT     |    5 |       746 |              120 | 382
test   | CL     |    5 |      1339 |              114 | 942
test   | IT     |    5 |       858 |              159 | 420

label counts:  label | train | dev
  Fact                        4098     444
  Issue                        107      15
  ArgumentPetitioner          1613     400
  ArgumentRespondent          1348     272
  Statute                     1099     120
  Dissent                      284   

## 5. Tokenise sentences and build windows
Each sentence is normalised exactly as in Phase 1, tokenised and capped at `SENT_MAX_TOKENS`; an empty sentence becomes a single separator token so it still gets a prediction. A window is a run of whole sentences, each followed by the separator (EOS) token, up to `WINDOW_TOK` tokens. Windows overlap by 50% so every sentence is seen with context on both sides; predictions are averaged afterwards.

In [70]:
def prep(doc):
    sents = [tokenizer.encode(normalize_text(s, False))[:SENT_MAX_TOKENS] or [EOS_ID] for s in doc["raw_sents"]]
    out = dict(doc); out["sents"] = sents
    out["y"] = np.array([L2I[l] for l in doc["labels"]], dtype=np.int64) if doc["labels"] is not None else None
    return out

t0 = time.time()
DOCS = {sp: [prep(d) for d in RAW[sp]] for sp in RAW}
tl = [len(s) for d in DOCS["train"] for s in d["sents"]]
print(f"tokenised {sum(len(v) for v in DOCS.values())} documents in {time.time() - t0:.0f}s | tokens/sentence: median {int(np.median(tl))}, p95 {int(np.percentile(tl, 95))}, "
      f"capped at {SENT_MAX_TOKENS} ({np.mean([t >= SENT_MAX_TOKENS for t in tl]):.1%} of sentences hit the cap)")

def make_windows(doc, overlap_frac):
    n = len(doc["sents"])
    if n == 0: return []
    if CONTEXT_MODE == "sentence": return [(i, i + 1) for i in range(n)]
    lens = [len(s) + 1 for s in doc["sents"]]
    wins, st = [], 0
    while st < n:
        tot, en = 0, st
        while en < n and (tot + lens[en] <= WINDOW_TOK or en == st):
            tot += lens[en]; en += 1
        wins.append((st, en))
        if en >= n: break
        st = max(st + 1, st + int((en - st) * (1 - overlap_frac)))
    return wins

def collate(items):
    """items: [(doc, start, end)] -> tensors for one batch of windows."""
    B = len(items)
    toks = [[t for j in range(st, en) for t in (doc["sents"][j] + [EOS_ID])] for doc, st, en in items]
    T = max(len(t) for t in toks); S = max(en - st for _, st, en in items)
    ids = torch.full((B, T), EOS_ID, dtype=torch.long)
    tok_sent = torch.full((B, T), S, dtype=torch.long)            # S = "ignore" bucket (separators, padding)
    sep_pos = torch.zeros((B, S), dtype=torch.long); valid = torch.zeros((B, S), dtype=torch.bool)
    cnt = torch.ones((B, S)); lab = torch.full((B, S), -100, dtype=torch.long); dom = torch.zeros(B, dtype=torch.long)
    teach = torch.zeros((B, S, K))
    for b, (doc, st, en) in enumerate(items):
        pos = 0
        for j in range(st, en):
            s = doc["sents"][j]; i = j - st
            ids[b, pos:pos + len(s)] = torch.tensor(s); tok_sent[b, pos:pos + len(s)] = i
            cnt[b, i] = len(s); sep_pos[b, i] = pos + len(s); valid[b, i] = True; pos += len(s) + 1
        if doc["y"] is not None: lab[b, :en - st] = torch.from_numpy(doc["y"][st:en])
        if doc.get("teacher") is not None: teach[b, :en - st] = torch.from_numpy(doc["teacher"][st:en])
        dom[b] = 0 if doc["domain"] == "CL" else 1
    return dict(ids=ids.to(device), tok_sent=tok_sent.to(device), sep_pos=sep_pos.to(device), valid=valid.to(device),
                cnt=cnt.to(device), lab=lab.to(device), dom=dom.to(device), teach=teach.to(device),
                n_sent=valid.sum(1))

def pool_sentences(h, batch):
    """h (B,T,D) -> sentence vectors (B,S,2D): mean of the sentence's tokens + the state at its separator token."""
    B, T, D = h.shape; S = batch["valid"].size(1)
    sums = torch.zeros(B, S + 1, D, dtype=h.dtype, device=h.device).scatter_add_(1, batch["tok_sent"][..., None].expand(-1, -1, D), h)[:, :S]
    mean = sums / batch["cnt"][..., None].to(h.dtype)
    sep = h.gather(1, batch["sep_pos"][..., None].expand(-1, -1, D))
    return torch.cat([mean, sep], -1)

nw = [len(make_windows(d, WIN_OVERLAP)) for d in DOCS["train"]]
print(f"windows: {sum(nw)} for {len(nw)} training docs (median {int(np.median(nw))} per doc) | context mode: {CONTEXT_MODE}")

tokenised 100 documents in 1s | tokens/sentence: median 34, p95 94, capped at 96 (4.7% of sentences hit the cap)
windows: 1326 for 80 training docs (median 10 per doc) | context mode: document


## 6. Metrics, decoding and the window-merging oracle test
* Macro-F1 is reported three ways, because the official scorer's convention is not stated: over labels present in gold (used for epoch selection), over the union of gold and predicted labels (scikit-learn's default), and over all 13 labels.
* **Oracle test:** the real window builder and merger are run with a stub model that outputs the gold label for each sentence. The result must be exactly the gold labels with the right lengths for every document; this checks the windowing, overlap-averaging and ordering on the real data, including very long documents.

In [71]:
def macro_scores(gold, pred):
    g = np.concatenate(gold) if len(gold) else np.zeros(0, dtype=np.int64); p = np.concatenate(pred) if len(pred) else np.zeros(0, dtype=np.int64)
    per = {}
    for c, l in enumerate(LABELS):
        tp = int(((g == c) & (p == c)).sum()); fp = int(((g != c) & (p == c)).sum()); fn = int(((g == c) & (p != c)).sum())
        pr = tp / (tp + fp) if tp + fp else 0.0; rc = tp / (tp + fn) if tp + fn else 0.0
        per[l] = dict(p=pr, r=rc, f=2 * pr * rc / (pr + rc) if pr + rc else 0.0, support=int((g == c).sum()), pred=int((p == c).sum()))
    present = [l for l in LABELS if per[l]["support"] > 0]; union = [l for l in LABELS if per[l]["support"] > 0 or per[l]["pred"] > 0]
    m = lambda ls, k: float(np.mean([per[l][k] for l in ls])) if ls else 0.0
    return dict(macro_f1=m(present, "f"), macro_p=m(present, "p"), macro_r=m(present, "r"), macro_f1_union=m(union, "f"),
                macro_f1_all=m(LABELS, "f"), accuracy=float((g == p).mean()) if len(g) else 0.0, per_label=per)

def merge_doc(doc, window_fn, bs=8):
    """window_fn(list of (doc, st, en)) -> list of (S_i, K) log-prob arrays. Returns (n_sent, K) averaged over overlapping windows."""
    n = len(doc["sents"]); acc = np.zeros((n, K), np.float32); wsum = np.zeros(n, np.float32)
    if n == 0: return acc
    wins = make_windows(doc, WIN_OVERLAP)
    for i in range(0, len(wins), bs):
        chunk = [(doc, st, en) for st, en in wins[i:i + bs]]
        for (_, st, en), lp in zip(chunk, window_fn(chunk)):
            S = en - st; w = np.minimum(np.arange(1, S + 1), np.arange(S, 0, -1)).astype(np.float32)   # trust window centres
            acc[st:en] += lp * w[:, None]; wsum[st:en] += w
    return acc / np.maximum(wsum, 1e-6)[:, None]

def oracle_window_fn(chunk):
    out = []
    for doc, st, en in chunk:
        lp = np.full((en - st, K), -20.0, dtype=np.float32); lp[np.arange(en - st), doc["y"][st:en]] = 0.0; out.append(lp)
    return out

for sp in ("train", "dev"):
    ok = all((merge_doc(d, oracle_window_fn).argmax(1) == d["y"]).all() and len(merge_doc(d, oracle_window_fn)) == len(d["sents"]) for d in DOCS[sp])
    print(f"oracle merge test on {sp}: {'PASS (windowing + overlap-averaging reproduce gold for every document)' if ok else 'FAIL'}")
    assert ok
nwin_max = max(len(make_windows(d, WIN_OVERLAP)) for sp in DOCS for d in DOCS[sp]); print("max windows in one document:", nwin_max)

oracle merge test on train: PASS (windowing + overlap-averaging reproduce gold for every document)
oracle merge test on dev: PASS (windowing + overlap-averaging reproduce gold for every document)
max windows in one document: 144


## 7. The model: Phase-1 backbone + adaptation technique + sentence head
* **Backbone:** the Phase-1 model as a feature extractor (embedding -> layers -> final RMSNorm).
* **LoRA / rsLoRA:** `W x + s·B A x` on every linear layer (`s = alpha/r`, or `alpha/sqrt(r)` for rsLoRA).
* **LORMA (my reconstruction, see the note at the top):** `(I + s·B A) W0 x` or `W0 (I + s·B A) x`, zero-initialised `B`.
* **IA3:** learned vectors that rescale attention keys and values (output side) and the FFN hidden activation (input of the down projection); initialised to 1 so training starts from the Phase-1 function.
* **Prompt tuning:** learned soft tokens prepended to every window; only they (and the head) are trained.
* **Knowledge distillation:** the student keeps the odd-numbered layers of the Phase-1 model (half depth, still initialised from Phase-1 weights) and is fully fine-tuned with `(1-α)·CE(gold) + α·T²·KL(teacher‖student)` computed per sentence.
* **Head (all runs):** sentence vector (2D) -> linear to `SENT_PROJ` + domain embedding (CL/IT) -> BiGRU over the sentences of the window -> dropout -> linear to 13 roles.

In [72]:
class LoRALinear(nn.Module):
    def __init__(self, base, r, alpha, dropout, rs=False):
        super().__init__()
        self.base, self.scale = base, (alpha / math.sqrt(r) if rs else alpha / r)
        self.A = nn.Parameter(torch.empty(r, base.in_features)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(base.out_features, r)); self.drop = nn.Dropout(dropout)
    def forward(self, x):
        return self.base(x) + (self.drop(x) @ self.A.t() @ self.B.t()) * self.scale

class LoRMALinear(nn.Module):
    """Multiplicative low-rank adaptation (my reconstruction of LORMA; verify against reference [12]).
    side='out': y = (I + s B A) (W0 x)      side='in': y = W0 ((I + s B A) x)       B = 0 at init => identity update."""
    def __init__(self, base, r, alpha, dropout, side="auto"):
        super().__init__()
        self.base, self.scale = base, alpha / r
        self.side = side if side != "auto" else ("out" if base.out_features <= base.in_features else "in")
        d = base.out_features if self.side == "out" else base.in_features
        self.A = nn.Parameter(torch.empty(r, d)); nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        self.B = nn.Parameter(torch.zeros(d, r)); self.drop = nn.Dropout(dropout)
    def forward(self, x):
        if self.side == "out":
            y = self.base(x)
            return y + (self.drop(y) @ self.A.t() @ self.B.t()) * self.scale
        return self.base(x + (self.drop(x) @ self.A.t() @ self.B.t()) * self.scale)

class IA3Linear(nn.Module):
    def __init__(self, base, mode):
        super().__init__()
        self.base, self.mode = base, mode
        self.l = nn.Parameter(torch.ones(base.out_features if mode == "out" else base.in_features))
    def forward(self, x):
        return self.base(x) * self.l if self.mode == "out" else self.base(x * self.l)

LINEAR_TARGETS = {"in_proj", "out_proj", "q", "k", "v", "o", "gate", "up", "down"}
def inject(module, make):
    n = 0
    for name, child in list(module.named_children()):
        new = make(name, child) if isinstance(child, nn.Linear) else None
        if new is not None: setattr(module, name, new); n += 1
        else: n += inject(child, make)
    return n

class Phase3Model(nn.Module):
    def __init__(self, backbone, exp):
        super().__init__()
        self.backbone, self.method, self.P = backbone, exp["method"], 0
        d = backbone_dim(backbone)
        if self.method != "full":
            for p in backbone.parameters(): p.requires_grad = False
        self.n_wrapped = 0
        if self.method in ("lora", "rslora"):
            self.n_wrapped = inject(backbone, lambda n, c: LoRALinear(c, exp["r"], exp["alpha"], exp["dropout"], rs=self.method == "rslora") if n in LINEAR_TARGETS else None)
        elif self.method == "lorma":
            self.n_wrapped = inject(backbone, lambda n, c: LoRMALinear(c, exp["r"], exp["alpha"], exp["dropout"], exp.get("side", "auto")) if n in LINEAR_TARGETS else None)
        elif self.method == "ia3":
            self.n_wrapped = inject(backbone, lambda n, c: IA3Linear(c, "out") if n in ("k", "v") else (IA3Linear(c, "in") if n == "down" else None))
        elif self.method == "prompt":
            self.P = exp["prompt_len"]
            ids = torch.randint(256, min(backbone.tok_emb.num_embeddings, 4000), (self.P,))
            self.prompt = nn.Parameter(backbone.tok_emb.weight[ids].detach().clone())
        self.proj = nn.Linear(2 * d, SENT_PROJ); self.dom = nn.Embedding(2, SENT_PROJ)
        self.gru = nn.GRU(SENT_PROJ, GRU_HIDDEN, batch_first=True, bidirectional=True) if CONTEXT_MODE == "document" else None
        self.drop = nn.Dropout(HEAD_DROPOUT)
        self.cls = nn.Linear(2 * GRU_HIDDEN if self.gru is not None else SENT_PROJ, K)

    def hidden(self, ids):
        m = self.backbone; x = m.tok_emb(ids)
        if self.P: x = torch.cat([self.prompt.unsqueeze(0).expand(x.size(0), -1, -1).to(x.dtype), x], 1)
        h = run_blocks(m, x)
        return h[:, self.P:] if self.P else h

    def forward(self, batch):
        sv = pool_sentences(self.hidden(batch["ids"]), batch)
        x = F.gelu(self.proj(sv)) + self.dom(batch["dom"])[:, None, :]
        if self.gru is not None:
            pk = nn.utils.rnn.pack_padded_sequence(x, batch["n_sent"], batch_first=True, enforce_sorted=False)
            out, _ = self.gru(pk)
            x, _ = nn.utils.rnn.pad_packed_sequence(out, batch_first=True, total_length=sv.size(1))
        return self.cls(self.drop(x))

def build_model():
    bb = build_backbone()
    if EXP.get("student_keep") == "odd": prune_layers(bb, list(range(1, len(bb.blocks), 2)))
    return Phase3Model(bb, EXP).to(device)

def count_params(m):
    return sum(p.numel() for p in m.parameters()), sum(p.numel() for p in m.parameters() if p.requires_grad)

_m = build_model(); TOTAL_P, TRAIN_P = count_params(_m)
extra = f" | {_m.n_wrapped} layers adapted" if _m.n_wrapped else ""
if hasattr(_m.backbone, "c") and hasattr(_m.backbone.c, "pattern"):
    extra += f" | layer pattern {''.join(p1_cfg.pattern[i] for i in range(len(p1_cfg.pattern)) if i % 2 == 1)}" if EXP.get("student_keep") == "odd" else f" | layer pattern {p1_cfg.pattern}"
print(f"{EXPERIMENT}: total {TOTAL_P / 1e6:.2f}M params | trainable {TRAIN_P / 1e6:.2f}M ({100 * TRAIN_P / TOTAL_P:.1f}%) | {len(_m.backbone.blocks)} backbone layers{extra}")
if WINDOW_TOK + _m.P > p1_cfg.n_ctx: raise ValueError(f"window {WINDOW_TOK} + prompt {_m.P} exceeds the model context {p1_cfg.n_ctx}")
del _m

D4_lora: total 126.35M params | trainable 5.02M (4.0%) | 12 backbone layers | 84 layers adapted


## 8. Training, evaluation, knowledge-distillation teacher loading
* Windows are rebuilt and shuffled every epoch; the loss is cross-entropy over sentences with class weights (`sqrt_inv`), plus the KD term for `*_kd` presets.
* After every epoch the dev set (CL+IT) is scored with macro-F1; the best epoch is kept and reused for the final retrain.
* For KD presets the teacher's per-sentence log-probabilities are loaded from `TEACHER_PKL`; train documents use its `train` entries and, in the final retrain, dev documents use its `dev` entries (teacher outputs on documents the teacher never trained on).

In [73]:
def class_weights(docs):
    c = np.bincount(np.concatenate([d["y"] for d in docs]), minlength=K).astype(np.float64)
    w = np.ones(K)
    if CLASS_WEIGHT == "sqrt_inv":
        ref = np.median(c[c > 0]) if (c > 0).any() else 1.0
        w = np.clip(np.sqrt(ref / np.maximum(c, 1.0)), 0.25, 5.0)
    return torch.tensor(w, dtype=torch.float32, device=device)

TEACHER = None
if EXP.get("kd_alpha") is not None:
    with open(TEACHER_PKL, "rb") as f: TEACHER = pickle.load(f)
    assert TEACHER["labels"] == LABELS, "teacher was produced with a different label vocabulary"
    for sp in ("train", "dev"):
        miss = [d["key"] for d in DOCS[sp] if d["key"] not in TEACHER[sp]]
        assert not miss, f"teacher file lacks {len(miss)} {sp} documents, e.g. {miss[:3]}"
    print(f"teacher: {TEACHER['meta']['experiment']} ({TEACHER['meta']['total_params'] / 1e6:.1f}M params, dev macro-F1 {TEACHER['meta']['dev_macro_f1']:.4f}) -> student {EXP['desc'][:60]}...")

def attach_teacher(docs, split):
    for d in docs: d["teacher"] = TEACHER[split][d["key"]].astype(np.float32)

def kd_loss(logits, batch, T):
    mask = batch["lab"] != -100
    ls = F.log_softmax(logits.float() / T, -1); pt = F.softmax(batch["teach"] / T, -1)
    return (F.kl_div(ls, pt, reduction="none").sum(-1)[mask]).mean() * T * T

@torch.no_grad()
def sent_logprobs(model, doc):
    model.eval()
    def window_fn(chunk):
        b = collate(chunk)
        with torch.autocast(device, dtype=torch.float16, enabled=device == "cuda"):
            lg = model(b)
        lp = F.log_softmax(lg.float(), -1).cpu().numpy()
        return [lp[i, :en - st] for i, (_, st, en) in enumerate(chunk)]
    return merge_doc(doc, window_fn, bs=MICRO_BS if CONTEXT_MODE == "document" else MICRO_BS_SENT)

def evaluate(model, docs):
    t0 = time.time(); lps = {d["key"]: sent_logprobs(model, d) for d in docs}
    sc = macro_scores([d["y"] for d in docs], [lps[d["key"]].argmax(1) for d in docs])
    by_dom = {dom: macro_scores([d["y"] for d in docs if d["domain"] == dom], [lps[d["key"]].argmax(1) for d in docs if d["domain"] == dom])["macro_f1"] for dom in ("CL", "IT")}
    return sc, by_dom, lps, time.time() - t0

def run_training(train_docs, dev_docs, epochs, tag, use_kd):
    model = build_model(); cw = class_weights(train_docs)
    bb, hd = [], []
    for n, p in model.named_parameters():
        if p.requires_grad: (hd if n.startswith(("proj", "dom", "gru", "cls")) else bb).append((n, p))
    def groups(lst, lr):
        return [{"params": [p for n, p in lst if p.dim() >= 2], "lr": lr, "base_lr": lr, "weight_decay": WEIGHT_DECAY},
                {"params": [p for n, p in lst if p.dim() < 2], "lr": lr, "base_lr": lr, "weight_decay": 0.0}]
    opt = torch.optim.AdamW([g for g in groups(bb, EXP["lr_backbone"]) + groups(hd, EXP["lr_head"]) if g["params"]], betas=(0.9, 0.98))
    scaler = torch.amp.GradScaler(device, enabled=device == "cuda")
    bs = MICRO_BS if CONTEXT_MODE == "document" else MICRO_BS_SENT; per_step = bs * GRAD_ACCUM
    def epoch_items(rng):
        items = [(d, st, en) for d in train_docs for st, en in make_windows(d, WIN_OVERLAP)]
        rng.shuffle(items); return items
    rng = random.Random(SEED); steps_per_epoch = max(1, len(epoch_items(random.Random(SEED))) // per_step)
    total = steps_per_epoch * epochs; warm = max(1, int(WARMUP_FRAC * total)); step = 0
    trainable = {n for n, p in model.named_parameters() if p.requires_grad}
    best = dict(f1=-1.0, epoch=0, state=None); hist = []; t_start = time.time()
    if device == "cuda": torch.cuda.reset_peak_memory_stats()
    for ep in range(1, epochs + 1):
        model.train(); items = epoch_items(rng); run, nb = 0.0, 0; t_ep = time.time()
        for i in range(0, len(items) - per_step + 1, per_step):
            f = (step + 1) / warm if step < warm else 0.05 + 0.95 * 0.5 * (1 + math.cos(math.pi * (step - warm) / max(1, total - warm)))
            for g in opt.param_groups: g["lr"] = g["base_lr"] * f
            for a in range(GRAD_ACCUM):
                b = collate(items[i + a * bs: i + (a + 1) * bs])
                with torch.autocast(device, dtype=torch.float16, enabled=device == "cuda"):
                    logits = model(b)
                loss = F.cross_entropy(logits.float().view(-1, K), b["lab"].view(-1), weight=cw, ignore_index=-100)
                if use_kd: loss = (1 - EXP["kd_alpha"]) * loss + EXP["kd_alpha"] * kd_loss(logits, b, EXP["kd_temp"])
                if not math.isfinite(loss.item()): continue
                scaler.scale(loss / GRAD_ACCUM).backward(); run += loss.item(); nb += 1
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_([p for _, p in bb + hd], CLIP)
            scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); step += 1
        rec = dict(epoch=ep, train_loss=run / max(nb, 1), sec=time.time() - t_ep)
        if dev_docs is not None:
            sc, by_dom, _, sec_eval = evaluate(model, dev_docs)
            rec.update(dev_macro_f1=sc["macro_f1"], dev_acc=sc["accuracy"], dev_cl=by_dom["CL"], dev_it=by_dom["IT"], eval_sec=sec_eval)
            if sc["macro_f1"] > best["f1"]:
                best = dict(f1=sc["macro_f1"], epoch=ep, state={n: v.detach().cpu().clone() for n, v in model.state_dict().items() if n in trainable})
            print(f"[{tag}] epoch {ep:2d}/{epochs} | loss {rec['train_loss']:.4f} | dev macro-F1 {sc['macro_f1']:.4f} (CL {by_dom['CL']:.3f} / IT {by_dom['IT']:.3f}) acc {sc['accuracy']:.3f} | {rec['sec']:.0f}s + eval {sec_eval:.0f}s")
        else:
            print(f"[{tag}] epoch {ep:2d}/{epochs} | loss {rec['train_loss']:.4f} | {rec['sec']:.0f}s")
        hist.append(rec)
        if (time.time() - t_start) / 60 > MAX_TRAIN_MINUTES and ep < epochs:
            print(f"[{tag}] time budget reached after epoch {ep}"); break
    if best["state"] is not None: model.load_state_dict(best["state"], strict=False)
    peak = torch.cuda.max_memory_allocated() / 1e9 if device == "cuda" else 0.0
    return model, dict(history=hist, best_epoch=best["epoch"] or len(hist), best_dev_f1=best["f1"], train_minutes=(time.time() - t_start) / 60, peak_gpu_gb=peak)

## 9. Phase A: train on the training split, select the epoch on the dev split

In [74]:
use_kd = TEACHER is not None
if use_kd:
    attach_teacher(DOCS["train"], "train"); attach_teacher(DOCS["dev"], "dev")
    if not ALLOW_LARGER_STUDENT: assert TOTAL_P < TEACHER["meta"]["total_params"], \
        f"student ({TOTAL_P/1e6:.1f}M) is not smaller than the teacher ({TEACHER['meta']['total_params']/1e6:.1f}M); pick a larger teacher or set ALLOW_LARGER_STUDENT=True"
model, infoA = run_training(DOCS["train"], DOCS["dev"], EPOCHS, f"{EXPERIMENT}/A", use_kd)
scA, by_domA, lpsA, sec_inf = evaluate(model, DOCS["dev"])
print(f"\nbest epoch {infoA['best_epoch']} | DEV macro-F1 {scA['macro_f1']:.4f} (union {scA['macro_f1_union']:.4f}, all-13 {scA['macro_f1_all']:.4f}) | "
      f"macro-P {scA['macro_p']:.4f} macro-R {scA['macro_r']:.4f} | accuracy {scA['accuracy']:.4f} | CL {by_domA['CL']:.4f} IT {by_domA['IT']:.4f} | dev inference {sec_inf:.0f}s")
print("\nrole                      |  P     R     F1   | support  predicted")
for l in LABELS:
    r = scA["per_label"][l]; print(f"  {l:24s}| {r['p']:.3f} {r['r']:.3f} {r['f']:.3f} | {r['support']:6d}  {r['pred']:6d}")

[D4_lora/A] epoch  1/12 | loss 1.8091 | dev macro-F1 0.4342 (CL 0.360 / IT 0.465) acc 0.500 | 100s + eval 5s
[D4_lora/A] epoch  2/12 | loss 1.0068 | dev macro-F1 0.4062 (CL 0.392 / IT 0.449) acc 0.519 | 100s + eval 5s
[D4_lora/A] epoch  3/12 | loss 0.5772 | dev macro-F1 0.5039 (CL 0.391 / IT 0.526) acc 0.551 | 100s + eval 5s
[D4_lora/A] epoch  4/12 | loss 0.3596 | dev macro-F1 0.5142 (CL 0.431 / IT 0.515) acc 0.582 | 100s + eval 5s
[D4_lora/A] epoch  5/12 | loss 0.2150 | dev macro-F1 0.5075 (CL 0.417 / IT 0.520) acc 0.567 | 100s + eval 5s
[D4_lora/A] epoch  6/12 | loss 0.1341 | dev macro-F1 0.4963 (CL 0.410 / IT 0.534) acc 0.589 | 100s + eval 5s
[D4_lora/A] epoch  7/12 | loss 0.0731 | dev macro-F1 0.4843 (CL 0.392 / IT 0.531) acc 0.542 | 100s + eval 5s
[D4_lora/A] epoch  8/12 | loss 0.0400 | dev macro-F1 0.5173 (CL 0.417 / IT 0.551) acc 0.571 | 100s + eval 5s
[D4_lora/A] epoch  9/12 | loss 0.0168 | dev macro-F1 0.5070 (CL 0.414 / IT 0.555) acc 0.570 | 100s + eval 5s
[D4_lora/A] epoch 1

### Error analysis on the dev split
Which role pairs get confused (gold -> predicted), and how the errors split by domain. The assignment asks specifically about closely related roles (for example the three precedent labels, petitioner vs respondent arguments, ratio vs ruling).

In [75]:
conf = Counter(); dom_err = Counter(); dom_n = Counter()
for d in DOCS["dev"]:
    p = lpsA[d["key"]].argmax(1)
    for g, q in zip(d["y"], p):
        dom_n[d["domain"]] += 1
        if g != q: conf[(LABELS[g], LABELS[q])] += 1; dom_err[d["domain"]] += 1
print("most frequent confusions (gold -> predicted, count):")
for (g, q), c in conf.most_common(10): print(f"   {g:24s} -> {q:24s} {c}")
print("error rate by domain:", {k: f"{dom_err[k] / max(dom_n[k], 1):.1%}" for k in dom_n})
pos = {}
for d in DOCS["dev"]:
    n = len(d["y"]); p = lpsA[d["key"]].argmax(1)
    for i, (g, q) in enumerate(zip(d["y"], p)): pos.setdefault(min(int(10 * i / max(n, 1)), 9), []).append(g == q)
print("accuracy by relative position in the document (deciles 0=start):", [f"{np.mean(pos[k]):.2f}" for k in sorted(pos)])

most frequent confusions (gold -> predicted, count):
   ArgumentPetitioner       -> RatioOfTheDecision       90
   Fact                     -> RatioOfTheDecision       85
   PrecedentReliedUpon      -> Fact                     61
   ArgumentRespondent       -> ArgumentPetitioner       54
   PrecedentReliedUpon      -> RatioOfTheDecision       54
   ArgumentRespondent       -> RatioOfTheDecision       53
   RulingByLowerCourt       -> RatioOfTheDecision       43
   ArgumentPetitioner       -> ArgumentRespondent       38
   ArgumentPetitioner       -> PrecedentReliedUpon      37
   Fact                     -> ArgumentPetitioner       30
error rate by domain: {'CL': '44.1%', 'IT': '40.8%'}
accuracy by relative position in the document (deciles 0=start): ['0.67', '0.67', '0.42', '0.65', '0.47', '0.56', '0.49', '0.59', '0.56', '0.62']


## 10. Save teacher outputs, Phase B (retrain on train+dev), predict the test set, write and validate the submission

In [76]:
def save_logprobs(path, splits, meta):
    out = {"labels": LABELS, "meta": meta}
    for name, docs in splits.items(): out[name] = {d["key"]: sent_logprobs(model, d) for d in docs}
    with open(path, "wb") as f: pickle.dump(out, f)
    return path

TEACHER_OUT = f"{EXP_DIR}/teacher_{EXPERIMENT}.pkl"
if SAVE_LOGPROBS and not use_kd:
    t0 = time.time()
    save_logprobs(TEACHER_OUT, {"train": DOCS["train"], "dev": DOCS["dev"], "test": DOCS["test"]},
                  dict(experiment=EXPERIMENT, total_params=TOTAL_P, dev_macro_f1=scA["macro_f1"], note="Phase-A model (trained on the train split only)"))
    print(f"saved teacher outputs -> {TEACHER_OUT} ({time.time() - t0:.0f}s). Use this path as TEACHER_PKL for a *_kd preset.")

if FINAL_RETRAIN:
    del model
    if device == "cuda": torch.cuda.empty_cache()
    final_epochs = max(3, infoA["best_epoch"])
    model, infoB = run_training(DOCS["train"] + DOCS["dev"], None, final_epochs, f"{EXPERIMENT}/B", use_kd)
else:
    infoB = dict(note="used the phase-A model (train split only)", best_epoch=infoA["best_epoch"])

t0 = time.time()
test_lps = {d["key"]: sent_logprobs(model, d) for d in DOCS["test"]}
infer_sec = time.time() - t0
submission = {d["id"]: [LABELS[i] for i in test_lps[d["key"]].argmax(1)] for d in DOCS["test"]}

# ---- validation of the file the scorer will read ----
assert len(submission) == len(DOCS["test"]), "duplicate document IDs"
for d in DOCS["test"]:
    pr = submission[d["id"]]
    assert len(pr) == len(d["raw_sents"]), f"doc {d['id']}: {len(pr)} predictions for {len(d['raw_sents'])} sentences"
    assert all(l in L2I for l in pr), f"doc {d['id']}: label outside label_vocab.json"
n_sent = sum(len(v) for v in submission.values())
dist = Counter(l for v in submission.values() for l in v)
print(f"test: {len(submission)} docs | {n_sent} sentences | every document has exactly one label per sentence | inference {infer_sec:.0f}s")
print("predicted distribution vs train distribution:")
for l in LABELS: print(f"   {l:24s} predicted {dist.get(l, 0) / n_sent:6.1%} | train {cnt['train'].get(l, 0) / sum(cnt['train'].values()):6.1%}")

PRED_PATH = f"{EXP_DIR}/prediction.json"; ZIP_PATH = f"{EXP_DIR}/submission_{EXPERIMENT}.zip"
json.dump(submission, open(PRED_PATH, "w"))
with zipfile.ZipFile(ZIP_PATH, "w", zipfile.ZIP_DEFLATED) as z: z.write(PRED_PATH, arcname="prediction.json")
with zipfile.ZipFile(ZIP_PATH) as z: print("submission archive:", ZIP_PATH, "->", z.namelist())
if SAVE_LOGPROBS:
    with open(f"{EXP_DIR}/final_test_logprobs_{EXPERIMENT}.pkl", "wb") as f:
        pickle.dump({"labels": LABELS, "meta": dict(experiment=EXPERIMENT, total_params=TOTAL_P), "test": test_lps}, f)

saved teacher outputs -> /kaggle/working/phase3_rr/D4_lora/teacher_D4_lora.pkl (47s). Use this path as TEACHER_PKL for a *_kd preset.
[D4_lora/B] epoch  1/8 | loss 1.7537 | 113s
[D4_lora/B] epoch  2/8 | loss 0.9961 | 113s
[D4_lora/B] epoch  3/8 | loss 0.6076 | 113s
[D4_lora/B] epoch  4/8 | loss 0.3302 | 113s
[D4_lora/B] epoch  5/8 | loss 0.1683 | 114s
[D4_lora/B] epoch  6/8 | loss 0.0716 | 114s
[D4_lora/B] epoch  7/8 | loss 0.0291 | 114s
[D4_lora/B] epoch  8/8 | loss 0.0133 | 113s
test: 10 docs | 2197 sentences | every document has exactly one label per sentence | inference 5s
predicted distribution vs train distribution:
   Fact                     predicted  19.5% | train  24.3%
   Issue                    predicted   0.6% | train   0.6%
   ArgumentPetitioner       predicted  10.9% | train   9.6%
   ArgumentRespondent       predicted   3.0% | train   8.0%
   Statute                  predicted   7.8% | train   6.5%
   Dissent                  predicted   0.0% | train   1.7%
   Precede

## 11. Save the experiment record and compare runs

In [77]:
result = dict(experiment=EXPERIMENT, description=EXP["desc"], method=EXP["method"], total_params=TOTAL_P, trainable_params=TRAIN_P,
              trainable_pct=100 * TRAIN_P / TOTAL_P, backbone_layers=len(model.backbone.blocks),
              phaseA=dict(best_epoch=infoA["best_epoch"], dev_macro_f1=scA["macro_f1"], dev_macro_f1_union=scA["macro_f1_union"],
                          dev_macro_f1_all=scA["macro_f1_all"], dev_macro_p=scA["macro_p"], dev_macro_r=scA["macro_r"],
                          dev_accuracy=scA["accuracy"], dev_cl=by_domA["CL"], dev_it=by_domA["IT"], per_label=scA["per_label"],
                          train_minutes=infoA["train_minutes"], peak_gpu_gb=infoA["peak_gpu_gb"], history=infoA["history"],
                          top_confusions=[(g, q, c) for (g, q), c in conf.most_common(10)]),
              phaseB=dict(infoB, test_inference_sec=infer_sec),
              config=dict(context_mode=CONTEXT_MODE, window_tok=WINDOW_TOK, win_overlap=WIN_OVERLAP, epochs=EPOCHS, class_weight=CLASS_WEIGHT,
                          lr_backbone=EXP["lr_backbone"], lr_head=EXP["lr_head"], kd=(EXP.get("kd_alpha"), EXP.get("kd_temp"))))
json.dump(result, open(f"{OUTPUT_DIR}/results_{EXPERIMENT}.json", "w"), indent=1, default=float)

rows = []
for p in sorted(glob.glob(f"{OUTPUT_DIR}/results_*.json")):
    r = json.load(open(p)); a = r["phaseA"]
    rows.append((r["experiment"], r["method"], r["total_params"] / 1e6, r["trainable_params"] / 1e6, a["dev_macro_f1"], a["dev_cl"], a["dev_it"], a["train_minutes"], a["peak_gpu_gb"]))
print("\nexperiment | method  | total M | trainable M | dev macro-F1 | CL     | IT     | train min | peak GB")
for r in rows: print(f"{r[0]:10s} | {r[1]:7s} | {r[2]:7.1f} | {r[3]:11.2f} | {r[4]:12.4f} | {r[5]:.4f} | {r[6]:.4f} | {r[7]:9.1f} | {r[8]:.1f}")
print("\nUpload", ZIP_PATH, "to Codabench (Phase 3), renamed to submission.zip.")


experiment | method  | total M | trainable M | dev macro-F1 | CL     | IT     | train min | peak GB
D2_ia3     | ia3     |   123.3 |        2.02 |       0.5280 | 0.4440 | 0.5374 |      16.8 | 2.7
D3_kd      | full    |    74.9 |       74.93 |       0.3078 | 0.3106 | 0.2896 |       9.7 | 3.1
D4_lora    | lora    |   126.4 |        5.02 |       0.5173 | 0.4166 | 0.5511 |      20.9 | 3.7

Upload /kaggle/working/phase3_rr/D4_lora/submission_D4_lora.zip to Codabench (Phase 3), renamed to submission.zip.


## Notes for the report and next steps
* All presets share data, windows, head, class weights, decoding and epoch budget, so differences come from the adaptation technique alone. Trainable parameters, training minutes, peak GPU memory, per-role and per-domain F1 are saved in `results_*.json`.
* Ablations that need no new technique (they do **not** count toward the six): `CONTEXT_MODE="sentence"` (does document context help?), `CLASS_WEIGHT="none"`, `WIN_OVERLAP`, `SENT_MAX_TOKENS`.
* **Knowledge distillation:** run a non-KD preset first (the strongest one is the natural teacher), point `TEACHER_PKL` at its `teacher_<EXP>.pkl`, then run the `*_kd` preset. Because the teacher file is per-sentence, a teacher from the Mamba notebook works too (cross-architecture distillation), as long as the student is smaller than the teacher.
* The six submissions must involve different Phase-1 models and include **LORMA** and **knowledge distillation**; this notebook covers one model's share. Confirm the LORMA formulation against reference [12] before reporting it.
* Teacher outputs on training documents are in-sample (the teacher saw those labels), so they are over-confident; the temperature `kd_temp` softens them. Mention this in the analysis.